# CPI Decomposition & Inflation Dynamics  
## Notebook 02: Data Cleaning & Feature Engineering  

---

### Objective

- Handle missing values
- Detect and treat outliers
- Standardize numerical features
- Create inflation metrics (MoM, YoY)
- Generate lag features for modeling
- Save clean dataset for EDA and modeling


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.max_columns", None)

print("Libraries imported successfully ✅")


Libraries imported successfully ✅


In [2]:
file_path = "../data/processed/processed_inflation.csv"

df = pd.read_csv(file_path)

df.head()


,Unnamed: 0.1,Unnamed: 0,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,0,0,1913.0,9.8,9.8,9.8,9.8,9.7,9.8,9.9,9.9,10.0,10.0,10.1,10.0
1,1,1,1914.0,10.0,9.9,9.9,9.8,9.9,9.9,10.0,10.2,10.2,10.1,10.2,10.1
2,2,2,1915.0,10.1,10.0,9.9,10.0,10.1,10.1,10.1,10.1,10.1,10.2,10.3,10.3
3,3,3,1916.0,10.4,10.4,10.5,10.6,10.7,10.8,10.8,10.9,11.1,11.3,11.5,11.6
4,4,4,1917.0,11.7,12.0,12.0,12.6,12.8,13.0,12.8,13.0,13.3,13.5,13.5,13.7


In [3]:
if "Date" in df.columns:
    df["Date"] = pd.to_datetime(df["Date"])
    df = df.sort_values("Date")
    df = df.set_index("Date")

print("Time index verified ✅")


Time index verified ✅


In [4]:
if "CPI" in df.columns:
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df["CPI"], label="CPI Index", color="blue")
    plt.title("Consumer Price Index (CPI) Over Time")
    plt.xlabel("Date")
    plt.ylabel("CPI Index")
    plt.grid(True)
    plt.legend()
    plt.show()

In [5]:
print("Missing values before cleaning:")
print(df.isnull().sum())

df = df.ffill()

df = df.bfill()

print("\nMissing values after cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
Unnamed: 0.1    0
Unnamed: 0      0
Year            0
Jan             0
Feb             0
Mar             0
Apr             0
May             1
Jun             1
Jul             1
Aug             1
Sep             1
Oct             1
Nov             1
Dec             1
dtype: int64

Missing values after cleaning:
Unnamed: 0.1    0
Unnamed: 0      0
Year            0
Jan             0
Feb             0
Mar             0
Apr             0
May             0
Jun             0
Jul             0
Aug             0
Sep             0
Oct             0
Nov             0
Dec             0
dtype: int64


In [6]:
def remove_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    
    return data[(data[column] >= lower) & (data[column] <= upper)]

if "CPI" in df.columns:
    df = remove_outliers_iqr(df, "CPI")
    print("Outliers removed from CPI column ✅")


In [7]:
if "CPI" in df.columns:
    df["Inflation_MoM"] = df["CPI"].pct_change() * 100
    print("Monthly inflation created ✅")
    
if "CPI" in df.columns:
    df["Inflation_YoY"] = df["CPI"].pct_change(12) * 100
    print("Year-over-Year inflation created ✅")

df.head()


,Unnamed: 0.1,Unnamed: 0,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,0,0,1913.0,9.8,9.8,9.8,9.8,9.7,9.8,9.9,9.9,10.0,10.0,10.1,10.0
1,1,1,1914.0,10.0,9.9,9.9,9.8,9.9,9.9,10.0,10.2,10.2,10.1,10.2,10.1
2,2,2,1915.0,10.1,10.0,9.9,10.0,10.1,10.1,10.1,10.1,10.1,10.2,10.3,10.3
3,3,3,1916.0,10.4,10.4,10.5,10.6,10.7,10.8,10.8,10.9,11.1,11.3,11.5,11.6
4,4,4,1917.0,11.7,12.0,12.0,12.6,12.8,13.0,12.8,13.0,13.3,13.5,13.5,13.7


In [8]:
df = df.drop(columns=["Unnamed: 0.1", "Unnamed: 0"], errors="ignore")

print("Cleaned extra columns")
df.head()

Cleaned extra columns


,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1913.0,9.8,9.8,9.8,9.8,9.7,9.8,9.9,9.9,10.0,10.0,10.1,10.0
1,1914.0,10.0,9.9,9.9,9.8,9.9,9.9,10.0,10.2,10.2,10.1,10.2,10.1
2,1915.0,10.1,10.0,9.9,10.0,10.1,10.1,10.1,10.1,10.1,10.2,10.3,10.3
3,1916.0,10.4,10.4,10.5,10.6,10.7,10.8,10.8,10.9,11.1,11.3,11.5,11.6
4,1917.0,11.7,12.0,12.0,12.6,12.8,13.0,12.8,13.0,13.3,13.5,13.5,13.7


In [9]:
df_long = df.melt(
    id_vars=["Year"],
    var_name="Month",
    value_name="CPI"
)

print("Converted to long time-series format")
df_long.head()

Converted to long time-series format


,Year,Month,CPI
0,1913.0,Jan,9.8
1,1914.0,Jan,10.0
2,1915.0,Jan,10.1
3,1916.0,Jan,10.4
4,1917.0,Jan,11.7


In [10]:
month_map = {
    "Jan":1,"Feb":2,"Mar":3,"Apr":4,"May":5,"Jun":6,
    "Jul":7,"Aug":8,"Sep":9,"Oct":10,"Nov":11,"Dec":12
}

df_long["Month_Num"] = df_long["Month"].map(month_map)

df_long["DATE"] = pd.to_datetime(dict(
    year=df_long["Year"],
    month=df_long["Month_Num"],
    day=1
))

df_long = df_long.sort_values("DATE").reset_index(drop=True)

print("Datetime index created successfully ✅")
df_long.head()


Datetime index created successfully ✅


,Year,Month,CPI,Month_Num,DATE
0,1913.0,Jan,9.8,1,1913-01-01
1,1913.0,Feb,9.8,2,1913-02-01
2,1913.0,Mar,9.8,3,1913-03-01
3,1913.0,Apr,9.8,4,1913-04-01
4,1913.0,May,9.7,5,1913-05-01


In [11]:
df_long = df_long[["DATE", "CPI"]]

df_long.set_index("DATE", inplace=True)

print("Final time-series ready")
df_long.head()

Final time-series ready


,CPI
DATE,
1913-01-01,9.8
1913-02-01,9.8
1913-03-01,9.8
1913-04-01,9.8
1913-05-01,9.7


In [12]:
lags = [1, 3, 6, 12]

for lag in lags:
    df_long[f"CPI_LAG_{lag}"] = df_long["CPI"].shift(lag)

print("Lag features created successfully ✅")

df_long.head()

Lag features created successfully ✅


,CPI,CPI_LAG_1,CPI_LAG_3,CPI_LAG_6,CPI_LAG_12
DATE,,,,,
1913-01-01,9.8,NaN,NaN,NaN,NaN
1913-02-01,9.8,9.8,NaN,NaN,NaN
1913-03-01,9.8,9.8,NaN,NaN,NaN
1913-04-01,9.8,9.8,9.8,NaN,NaN
1913-05-01,9.7,9.8,9.8,NaN,NaN


In [13]:
df_long["Rolling_Mean_3"] = df_long["CPI"].rolling(window=3).mean()
df_long["Rolling_Mean_12"] = df_long["CPI"].rolling(window=12).mean()
df_long["Rolling_Std_12"] = df_long["CPI"].rolling(window=12).std()

print("Rolling features created successfully ✅")

df_long.head()

Rolling features created successfully ✅

,CPI,CPI_LAG_1,CPI_LAG_3,CPI_LAG_6,CPI_LAG_12,Rolling_Mean_3,Rolling_Mean_12,Rolling_Std_12
DATE,,,,,,,,
1913-01-01,9.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1913-02-01,9.8,9.8,NaN,NaN,NaN,NaN,NaN,NaN
1913-03-01,9.8,9.8,NaN,NaN,NaN,9.800000,NaN,NaN
1913-04-01,9.8,9.8,9.8,NaN,NaN,9.800000,NaN,NaN
1913-05-01,9.7,9.8,9.8,NaN,NaN,9.766667,NaN,NaN


In [14]:
df = df.dropna()

print("Final dataset shape:", df.shape)
df.head()

Final dataset shape: (110, 13)


,Year,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,1913.0,9.8,9.8,9.8,9.8,9.7,9.8,9.9,9.9,10.0,10.0,10.1,10.0
1,1914.0,10.0,9.9,9.9,9.8,9.9,9.9,10.0,10.2,10.2,10.1,10.2,10.1
2,1915.0,10.1,10.0,9.9,10.0,10.1,10.1,10.1,10.1,10.1,10.2,10.3,10.3
3,1916.0,10.4,10.4,10.5,10.6,10.7,10.8,10.8,10.9,11.1,11.3,11.5,11.6
4,1917.0,11.7,12.0,12.0,12.6,12.8,13.0,12.8,13.0,13.3,13.5,13.5,13.7


In [15]:
clean_path = "../data/processed/clean_inflation.csv"

df.to_csv(clean_path)

print("Clean dataset saved successfully ✅")


Clean dataset saved successfully ✅


##  Key Observations

- Missing values treated
- Outliers reduced
- Inflation metrics created
- Lag features generated
- Rolling statistics added
- Dataset ready for decomposition & modeling

------------------------------------------------------